In [17]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, GRU, Dropout, Dense
from tensorflow.keras.layers import BatchNormalization, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, make_scorer
from sklearn.base import BaseEstimator


In [2]:
#Get Stock Data from Yahoo Finance
stock="QQQ"
data = yf.download(stock, start="2010-01-01")
data.head()



[*********************100%%**********************]  1 of 1 completed


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2010-01-04,46.330002,46.490002,46.270000,46.419998,40.787140,62822800
2010-01-05,46.389999,46.500000,46.160000,46.419998,40.787140,62935600
2010-01-06,46.400002,46.549999,46.070000,46.139999,40.541126,96033000
2010-01-07,46.209999,46.270000,45.919998,46.169998,40.567459,77094100
2010-01-08,46.070000,46.549999,45.930000,46.549999,40.901356,88886600


In [3]:
#Check for missing values
data.isna().sum()

Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

In [4]:
#Add Technical Indicators
def add_technical_indicators(data):
    #Simple Moving Average
    data['SMA'] = data['Close'].rolling(window=20).mean()
    #Exponential Moving Average
    data['EMA'] = data['Close'].ewm(span=20, adjust=False).mean()
    #Momentum
    data['Momentum'] = data['Close']-1
    #Relative Strength Index
    delta = data['Close'].diff()
    gain = (delta.where(delta>0,0)).rolling(window=14).mean()
    loss = (-delta.where(delta<0,0)).rolling(window=14).mean()
    RS = gain/loss
    data['RSI'] = 100 - (100/(1+RS))
    #Average True Range
    data['ATR'] = data['High'] - data['Low']
    data['ATR'] = data['ATR'].rolling(window=14).mean()
    #Bollinger Bands
    data['MA20'] = data['Close'].rolling(window=20).mean()
    data['20dSTD'] = data['Close'].rolling(window=20).std()
    data['UpperBand'] = data['MA20'] + (data['20dSTD'] * 2)
    data['LowerBand'] = data['MA20'] - (data['20dSTD'] * 2)
    return data

data = add_technical_indicators(data)
data.dropna(inplace=True)
data.head()

,Open,High,Low,Close,Adj Close,Volume,SMA,EMA,Momentum,RSI,ATR,MA20,20dSTD,UpperBand,LowerBand
Date,,,,,,,,,,,,,,,
2010-02-01,42.900002,43.279999,42.880001,43.259998,38.010597,136827100,45.377499,44.982603,42.259998,30.179022,0.864285,45.377499,1.224998,47.827494,42.927504
2010-02-02,43.330002,43.779999,43.029999,43.650002,38.353260,110727700,45.239000,44.855689,42.650002,36.041967,0.874285,45.239000,1.257097,47.753193,42.724806
2010-02-03,43.439999,43.970001,43.419998,43.889999,38.564152,93581600,45.112500,44.763718,42.889999,33.150698,0.850714,45.112500,1.259293,47.631086,42.593914
2010-02-04,43.570000,43.660000,42.619999,42.619999,37.448250,151652500,44.936500,44.559554,41.619999,27.901528,0.903571,44.936500,1.350786,47.638072,42.234927
2010-02-05,42.730000,43.020000,42.119999,42.980000,37.764568,213618600,44.777000,44.409121,41.980000,32.814382,0.903571,44.777000,1.385364,47.547727,42.006273


In [5]:
# Feature Selection
correlation = data.corr()
relevant_features = correlation['Adj Close'].sort_values(ascending=False)[:5].index
data = data[relevant_features]

data.loc[:, 'Target'] = (data['Adj Close'].shift(-1) > data['Adj Close']).astype(int)
data = data.dropna()

def split_data(data, V):
    X = data.drop('Target', axis=1)
    y = data['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_data(data, 5)

C:\Users\eugen\AppData\Local\Temp\ipykernel_17176\975696700.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:, 'Target'] = (data['Adj Close'].shift(-1) > data['Adj Close']).astype(int)


In [6]:
# Scale the data
scalers = [MinMaxScaler(), StandardScaler(), RobustScaler()]
scaled_data = []
for scaler in scalers:
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_scaled = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
    X_test_scaled = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)
    scaled_data.append((X_train_scaled, X_test_scaled))

In [7]:
# Define the model with hyperparameters as arguments
def create_model(lstm_units, dropout_rate, learning_rate, regularizer):
    model = Sequential([
        Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),
        LSTM(lstm_units, activation='tanh', return_sequences=True, kernel_regularizer=regularizer),
        Dropout(dropout_rate),
        LSTM(lstm_units, activation='tanh', kernel_regularizer=regularizer),
        Dropout(dropout_rate),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [8]:
# Define the parameter grid for grid search
param_grid = {
    'lstm_units': [32, 64, 128],
    'dropout_rate': [0.1, 0.2, 0.3],
    'learning_rate': [0.001, 0.01, 0.1],
    'batch_size': [16, 32, 64],
    'regularizer': [l1_l2(l1=0.01, l2=0.01), l1_l2(l1=0.001, l2=0.001)]
}

In [18]:
# Define a custom scoring function
def custom_accuracy(y_true, y_pred):
    y_pred_classes = np.round(y_pred).astype(int)
    return accuracy_score(y_true, y_pred_classes)

# Create a scorer from the custom accuracy function
custom_scorer = make_scorer(custom_accuracy)

In [23]:
# Define a custom estimator class
class KerasClassifier(BaseEstimator):
    def __init__(self, lstm_units=64, dropout_rate=0.2, learning_rate=0.001, regularizer=None, batch_size=32):
        self.lstm_units = lstm_units
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.regularizer = regularizer
        self.batch_size = batch_size

    def fit(self, X, y):
        y_categorical = to_categorical(y)
        self.model = create_model(
            lstm_units=self.lstm_units,
            dropout_rate=self.dropout_rate,
            learning_rate=self.learning_rate,
            regularizer=self.regularizer
        )
        self.model.fit(X, y_categorical, batch_size=self.batch_size, epochs=1, verbose=0)
        return self

    def predict(self, X):
        y_pred_probabilities = self.model.predict(X)
        y_pred_classes = np.argmax(y_pred_probabilities, axis=1)
        return y_pred_classes

    def score(self, X, y):
        y_pred_probabilities = self.model.predict(X)
        return custom_accuracy(y, y_pred_probabilities)

    def set_params(self, **params):
        for param, value in params.items():
            setattr(self, param, value)
        return self


In [24]:
# Perform grid search for each scaled dataset
best_models = []
for X_train_scaled, X_test_scaled in scaled_data:
    model = KerasClassifier()
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring=custom_scorer, cv=5, verbose=2)
    grid_search_result = grid_search.fit(X_train_scaled, y_train)
    best_models.append(grid_search_result.best_estimator_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
[CV] END batch_size=16, dropout_rate=0.1, learning_rate=0.001, lstm_units=32, regularizer=<keras.src.regularizers.regularizers.L1L2 object at 0x000001497CAA2780>; total time=   3.5s
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
[CV] END batch_size=16, dropout_rate=0.1, learning_rate=0.001, lstm_units=32, regularizer=<keras.src.regularizers.regularizers.L1L2 object at 0x000001497CAA2780>; total time=   3.0s
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
[CV] END batch_size=16, dropout_rate=0.1, learning_rate=0.001, lstm_units=32, regularizer=<keras.src.regularizers.regularizers.L1L2 object at 0x000001497CAA2780>; total time=   3.3s
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
[CV] END batch_size=16, dropout_rate=0.1, learning_rate=0.001, lstm_units=32, regularizer=<keras.src.regularizers.regularizers.L1L2 object at 0x000001497CAA2780>; total time=   3.0s
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
[CV] END 

In [25]:
# Evaluate the best models on the test set
for i, (_, X_test_scaled) in enumerate(scaled_data):
    best_model = best_models[i]
    y_pred_classes = best_model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred_classes)
    print(f"Best Model {i+1} Test Accuracy: {accuracy}")

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Best Model 1 Test Accuracy: 0.5288326300984529
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Best Model 2 Test Accuracy: 0.4711673699015471
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Best Model 3 Test Accuracy: 0.5288326300984529
